# E14 CPU harness (Kaggle pinned image)

Unit tests + the fake-server harness of `Q27-XHIGH-E15_MTP_ENGINE_DUCK_SGLANG.ipynb` on the same image as the GPU notebook. Kit: `foysalemonshanto/q27-e14-cpu-kit` (q27x_e14_cpu_kit_f8cf18e547c9.bin).


In [ ]:
# E14 CPU harness on Kaggle (pinned image = the GPU notebook's): unit tests + the fake-server notebook harness
import hashlib, io, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
KIT_NAME, KIT_SHA = "q27x_e14_cpu_kit_f8cf18e547c9.bin", "f8cf18e547c93641ca339cfa03e436babe77ce47ee5efbde711c8fdbf77c9e2e"
TARGET = "nb/Q27-XHIGH-E15_MTP_ENGINE_DUCK_SGLANG.ipynb"
SCALES = [0.1]
TESTS = ["test_e14_measure.py", "test_e14_loadpath.py", "test_e14_sglang_patches.py", "test_e14_nccl_bench_and_trace.py"]
RUNS = [["harness_e14s_nb.py", "E14S_MODE", "e15"], ["harness_e14s_nb.py", "E14S_MODE", "rerun"]]
WH = "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"
OUT = Path("/kaggle/working"); K = Path("/tmp/e14_kit"); T0 = time.time()
REPORT = {"kit": KIT_NAME, "target": TARGET, "python": sys.version, "scales": SCALES, "pytest": None, "harness": {}}


def save():
    (OUT / "cpu_harness_report.json").write_text(json.dumps(REPORT, indent=1), encoding="utf-8")


def find(name, sha):
    for pat in (name, "*/" + name, "*/*/" + name, "*/*/*/" + name):
        for f in sorted(Path("/kaggle/input").glob(pat)):
            b = f.read_bytes()
            if hashlib.sha256(b).hexdigest() == sha:
                return b
    raise FileNotFoundError(f"{name} ({sha[:12]}) not under /kaggle/input: attach foysalemonshanto/q27-e14-cpu-kit")


shutil.rmtree(K, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(find(KIT_NAME, KIT_SHA))) as z:
    z.extractall(K)
man = json.loads((K / "MANIFEST.json").read_text())
bad = [k for k, h in man.items() if hashlib.sha256((K / k).read_bytes()).hexdigest() != h]
assert not bad, bad
print("kit OK:", len(man), "files |", sys.version.split()[0], flush=True)
# the GPU notebook's runtime pins (its CELL 1): transformers 4.57.6 + huggingface-hub 0.36.2 from the S49 wheelhouse
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WH,
                    "transformers==4.57.6", "huggingface-hub==0.36.2"], capture_output=True, text=True)
REPORT["pins"] = {"rc": r.returncode, "tail": (r.stdout + r.stderr)[-800:]}
try:
    import pytest  # noqa: F401
    REPORT["pytest_source"] = "image"
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", str(K / "wheels"),
                        "pytest"], capture_output=True, text=True)
    REPORT["pytest_source"] = {"kit wheels rc": r.returncode, "tail": (r.stdout + r.stderr)[-500:]}
print("pins:", REPORT["pins"]["rc"], "| pytest:", REPORT["pytest_source"], flush=True)
ENV = dict(os.environ, E14_SGL_SRC=str(K / "sgl_ref"), E14_FIXTURES_DIR=str(K / "fixtures"), PYTHONIOENCODING="utf-8")
t = time.time()
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--confcutdir", str(K / "src"),
                    "--rootdir", str(K / "src")] + TESTS, cwd=K / "src", env=ENV, capture_output=True, text=True,
                   timeout=3600)
(OUT / "pytest.log").write_text(r.stdout + r.stderr, encoding="utf-8")
REPORT["pytest"] = {"rc": r.returncode, "sec": round(time.time() - t), "summary": (r.stdout.strip().splitlines() or [""])[-1],
                    "failures": [l for l in r.stdout.splitlines() if l.startswith(("FAILED", "ERROR"))][:40]}
save()
print("pytest:", REPORT["pytest"]["summary"], "| rc", r.returncode, flush=True)
for sc in SCALES:
    for hfile, var, scen in RUNS:
        key = f"{scen}@{sc}"
        env = dict(ENV, E14_HARNESS_ROOT=f"/tmp/e14_h_{scen}_{sc}", **{var: scen})
        t = time.time()
        try:
            r = subprocess.run([sys.executable, str(K / "src/harness" / hfile), str(K / TARGET), str(sc)],
                               cwd=K / "src/harness", env=env, capture_output=True, text=True, timeout=7200)
            out = r.stdout + r.stderr
            rc = r.returncode
        except subprocess.TimeoutExpired as e:
            out, rc = (e.stdout or "") if isinstance(e.stdout, str) else "", "timeout"
        (OUT / f"harness_{scen}_{sc}.log").write_text(out, encoding="utf-8")
        lines = out.splitlines()
        REPORT["harness"][key] = {"rc": rc, "sec": round(time.time() - t), "ok": "@@NB_OK@@" in out,
                                  "summary": [l for l in lines if l.startswith("[e14")],
                                  "fails": [l[:1500] for l in lines if l.startswith("   FAIL")],
                                  "traceback_tail": out[-3000:] if "Traceback" in out else ""}
        save()
        print(key, "->", REPORT["harness"][key]["summary"], "ok" if REPORT["harness"][key]["ok"] else "FAIL",
              REPORT["harness"][key]["fails"][:6], flush=True)
REPORT["sec"] = round(time.time() - T0)
REPORT["all_ok"] = REPORT["pytest"]["rc"] == 0 and all(v["ok"] for v in REPORT["harness"].values())
save()
print("ALL OK" if REPORT["all_ok"] else "NOT OK", "| total", REPORT["sec"], "s", flush=True)
